In [1]:
import sys
sys.path.append('..')
from transformers import AutoModelForMaskedLM, AutoTokenizer
from prosst.structure.get_sst_seq import SSTPredictor
from Bio import SeqIO
import torch
import pandas as pd
from scipy.stats import spearmanr

/home/abhishekg/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


In [2]:
import os
os.environ["http_proxy"] = "http://127.0.0.1:15777"
os.environ["https_proxy"] = "http://127.0.0.1:15777"

Load ProSST from Hugging Face. 
(You may need to configure the proxy settings if you are in a region that cannot access the hugging face model.)

In [2]:
prosst_model = AutoModelForMaskedLM.from_pretrained("AI4Protein/ProSST-2048", trust_remote_code=True)
prosst_tokenizer = AutoTokenizer.from_pretrained("AI4Protein/ProSST-2048", trust_remote_code=True)

Loading weights: 100%|██████████| 275/275 [00:00<00:00, 17728.22it/s]
ProSSTForMaskedLM LOAD REPORT from: AI4Protein/ProSST-2048
Key                            | Status  | 
-------------------------------+---------+-
cls.predictions.decoder.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load strcuture quantizer

In [3]:
predictor = SSTPredictor(structure_vocab_size=2048)

---------- Load Model on cuda ----------
MODEL: 5.90M parameters


Read protein sequence

In [4]:
residue_sequence = str(SeqIO.read('example_data/GRB2_HUMAN_Faure_2021.fasta', 'fasta').seq)
    

Quantize the structure

In [5]:
from prosst.structure.get_sst_seq import SSTPredictor
predictor = SSTPredictor(structure_vocab_size=2048)
structure_sequence = predictor.predict_from_pdb('example_data/GRB2_HUMAN_Faure_2021.pdb')[0]['2048_sst_seq']

---------- Load Model on cuda ----------
MODEL: 5.90M parameters
---------- Building Subgraphs ----------


100%|██████████| 1/1 [00:00<00:00,  1.61it/s]


In [8]:
print(structure_sequence)

[{'name': 'GRB2_HUMAN_Faure_2021.pdb', 'aa_seq': 'MEAIAKYDFKATADDELSFKRGDILKVLNEECDQNWYKAELNGKDGFIPKNYIEMKPHPWFFGKIPRAKAEEMLSKQRHDGAFLIRESESAPGDFSLSVKFGNDVQHFKVLRDGAGKYFLWVVKFNSLNELVDYHRSTSVSRNQQIFLRDIEQVPQQPTYVQALFDFDPQEDGELGFRRGDFIHVMDNSDPNWWKGACHGQTGMFPRNYVTPVNRNV', '2048_sst_seq': [1211, 1599, 10, 1853, 1321, 1738, 919, 1863, 1863, 576, 99, 1396, 1484, 1484, 99, 1493, 886, 815, 815, 1863, 1863, 1863, 1877, 508, 1168, 1556, 1352, 339, 1493, 1007, 1860, 714, 139, 732, 139, 139, 295, 1556, 142, 694, 1561, 923, 1677, 74, 772, 142, 1321, 1853, 1544, 1447, 530, 138, 1574, 791, 1700, 1599, 1667, 1161, 1161, 1353, 849, 1285, 1914, 1473, 1308, 328, 92, 380, 49, 158, 524, 1827, 1672, 1891, 108, 1879, 1602, 1602, 91, 1261, 818, 850, 1407, 1353, 1203, 1203, 1463, 1854, 1860, 1413, 1984, 430, 430, 577, 1926, 1747, 159, 1947, 2018, 441, 1519, 782, 542, 552, 1891, 677, 1239, 93, 1168, 1713, 1853, 1321, 545, 1783, 1677, 826, 1298, 1830, 1200, 557, 577, 1186, 1575, 1653, 108, 308, 906, 453, 187, 47

In [7]:
structure_sequence = predictor.predict_from_pdb("example_data/GRB2_HUMAN_Faure_2021.pdb")[0]['2048_sst_seq']

---------- Building Subgraphs ----------


100%|██████████| 1/1 [00:00<00:00,  1.50it/s]


Shift the quantized structure sequence, (for 3 special tokens [CLS], [SEP] and [PAD])

In [6]:
structure_sequence_offset = [i + 3 for i in structure_sequence]

Prepare model input

In [7]:
tokenized_res = prosst_tokenizer([residue_sequence], return_tensors='pt')
input_ids = tokenized_res['input_ids']
attention_mask = tokenized_res['attention_mask']
structure_input_ids = torch.tensor([1, *structure_sequence_offset, 2], dtype=torch.long).unsqueeze(0)

Inferece 

In [8]:
with torch.no_grad():
    outputs = prosst_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        ss_input_ids=structure_input_ids
    )
logits = torch.log_softmax(outputs.logits[:, 1:-1], dim=-1).squeeze()

`use_return_dict` is deprecated! Use `return_dict` instead!


Score mutants

In [9]:
df = pd.read_csv("example_data/GRB2_HUMAN_Faure_2021.csv")
mutants = df['mutant'].tolist()

In [10]:
vocab = prosst_tokenizer.get_vocab()
pred_scores = []
for mutant in mutants:
    mutant_score = 0
    for sub_mutant in mutant.split(":"):
        wt, idx, mt = sub_mutant[0], int(sub_mutant[1:-1]) - 1, sub_mutant[-1]
        pred = logits[idx, vocab[mt]] - logits[idx, vocab[wt]]
        mutant_score += pred.item()
    pred_scores.append(mutant_score)

Compute the spearman correlation

In [11]:
spearmanr(pred_scores, df['DMS_score'])

SignificanceResult(statistic=0.00586571556644082, pvalue=0.13979980791346908)